# **Feature visualization: asking the network to draw what it looks for**

A practice for the module ["What the network has learned: features and concepts"](https://ai-interpretability.school).

In the lesson we asked the network the reverse question: not "how does this channel respond to
this picture" but **"which picture will make the channel respond most strongly"**. Here we ask
that question for real — and see why the naive answer to it is useless.

The notebook runs on a CPU: the two optimizations take about a quarter of a minute.

In [ ]:
import io
import urllib.request

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

torch.manual_seed(0)

SIZE = 160          # the side of the picture we are drawing
LAYER = 'layer4'    # the layer the channel of interest lives in
CHANNEL = 12        # the channel number — change it and see what gets drawn
STEPS = 384         # optimization steps

## 1. What we optimize

Everything is as in the lesson: the network is trained and frozen, only the picture at the input
changes. We need two things — a way to read the activation of the chosen channel, and a way to
reach it with a gradient.

We read the activation with a hook: it intercepts the output of the layer on the forward pass,
so we do not have to rewrite the network itself.

In [ ]:
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()
for p in model.parameters():
    p.requires_grad_(False)          # the weights are frozen: only the picture will change

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

captured = {}
getattr(model, LAYER).register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))


def activation(img, channel=CHANNEL):
    """Mean activation of the channel. Input is a picture in [0,1]; normalization inside."""
    model((img - MEAN) / STD)
    return captured['a'][0, channel].mean()


def show(img, title):
    plt.figure(figsize=(3, 3))
    plt.imshow(img[0].permute(1, 2, 0).clamp(0, 1).numpy())
    plt.title(title)
    plt.axis('off')
    plt.show()

## 2. The naive optimization

We do exactly what the lesson describes and nothing more: start from noise, step along the
gradient of the activation, repeat several hundred times.

The functions `jitter` and `total_variation` will be needed later — for now `optimize` is called
with both the shifts and the penalty switched off.

In [ ]:
def optimize(use_jitter, tv_weight, steps=STEPS, lr=0.08):
    """Gradient ascent on the input: we look for a picture the channel responds to more."""
    img = (torch.rand(1, 3, SIZE, SIZE) * 0.1 + 0.45).requires_grad_(True)
    opt = torch.optim.Adam([img], lr=lr)
    for _ in range(steps):
        view = jitter(img) if use_jitter else img
        loss = -activation(view) + tv_weight * total_variation(img)
        opt.zero_grad()
        loss.backward()                # the gradient is taken with respect to the picture, not the weights
        opt.step()
        with torch.no_grad():
            img.clamp_(0, 1)           # a picture has to stay a picture: values in [0,1]
    return img.detach()


def jitter(img, max_shift=12):
    """A shift and a small change of scale before the pass — transformation robustness."""
    dx, dy = torch.randint(-max_shift, max_shift + 1, (2,))
    img = torch.roll(img, (int(dy), int(dx)), dims=(2, 3))
    n = max(int(SIZE * (1 + 0.15 * (torch.rand(1).item() - 0.5))), 8)
    return F.interpolate(img, size=(n, n), mode='bilinear', align_corners=False)


def total_variation(img):
    """Total variation: the mean step between neighbouring pixels. The larger, the noisier."""
    return ((img[:, :, 1:, :] - img[:, :, :-1, :]).abs().mean()
            + (img[:, :, :, 1:] - img[:, :, :, :-1]).abs().mean())

In [ ]:
naive = optimize(use_jitter=False, tv_weight=0.0)
with torch.no_grad():
    a_naive = activation(naive).item()

show(naive, f'naive optimization, activation: {a_naive:.1f}')
print(f'channel activation: {a_naive:.2f}')
print(f'mean step between neighbouring pixels: {total_variation(naive).item():.3f}')

## 3. The check: we have found an adversarial example

The picture above looks like television static, and its activation is large. How large is not a
rhetorical question, and it can be answered exactly: let us compare it with real images, that
is, with the ones the network was trained for.

In [ ]:
tf = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])
FILES = ['data/cat.jpg', 'data/hog.jpg', 'data/pig.png', 'data/cat_and_dog.jpg',
         'assets/tim-foster-w-X64-Gjbclg-unsplash.jpg']


def load(path):
    raw = urllib.request.urlopen(f'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/' + path, timeout=30).read()
    return tf(Image.open(io.BytesIO(raw)).convert('RGB')).unsqueeze(0)


real = {path: load(path) for path in FILES}
with torch.no_grad():
    scores = {path: activation(img).item() for path, img in real.items()}

for path, value in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f'{value:7.2f}  {path}')

best = max(scores.values())
print(f'\nthe activation of noise is this many times above the best real picture: {a_naive / best:.0f}')

**Task 1.** The ratio came out in the tens. Check that it is not about this
particular channel: take three or four other numbers in the same layer and compute the ratio for
each. Does the order of magnitude hold?

What this means: the maximum of the activation lies where pictures from the real world do not
occur. The optimization did its job honestly — the honest maximum simply turned out to be
useless.

In [ ]:
# Your code here

## 4. What keeps the optimization inside the region of plausible pictures

The lesson names three devices, and two of them work here:

- **shifts and scaling** (`jitter`) — at every step the picture trembles slightly before the
  pass. Adversarial noise is fragile and is destroyed by a shift of a couple of pixels, while a
  robust pattern survives the trembling;
- **a penalty on steps** (`total_variation`) — we tell the optimization outright that smooth is
  better.

The third device, replacing the parameterization with a frequency one, we leave aside: it needs
a separate conversation about the spectrum, and the first two give an effect on their own.

In [ ]:
tamed = optimize(use_jitter=True, tv_weight=0.3)
with torch.no_grad():
    a_tamed = activation(tamed).item()

show(tamed, f'with shifts and a penalty, activation: {a_tamed:.1f}')
print(f'channel activation: {a_tamed:.2f}  (was {a_naive:.2f})')
print(f'mean step between neighbouring pixels: {total_variation(tamed).item():.3f}')

**Task 2.** The activation dropped and the picture became readable. That is not a
coincidence but a price: we narrowed the search region, and inside it the maximum is lower.

Work out which of the two devices does what. Run `optimize` twice more — with the shifts but
without the penalty, and with the penalty but without the shifts — and compare the four pictures
together with their activations and steps. Which of the devices removes the noise, and which
hardly matters?

In [ ]:
# Your code here

## 5. Next to the drawn one — the real ones

The lesson says: a useful practice is to look at the optimized picture together with dataset
examples. The optimization shows what the channel looks for "ideally"; real photographs show
what it finds in practice. **The gap between them is a diagnosis in itself.**

In [ ]:
top = sorted(scores.items(), key=lambda kv: -kv[1])[:3]

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(tamed[0].permute(1, 2, 0).clamp(0, 1).numpy())
axes[0].set_title(f'with shifts and a penalty, activation\n{a_tamed:.1f}')
for ax, (path, value) in zip(axes[1:], top):
    ax.imshow(real[path][0].permute(1, 2, 0).numpy())
    ax.set_title(f'{path.split("/")[-1]}\n{value:.2f}')
for ax in axes:
    ax.axis('off')
plt.show()

**Task 3.** Our set of real pictures is tiny — seven of them, and not one was
chosen for this channel. Take two or three images of your own by address and see what activates
the channel more strongly.

The question all of this was for: does the drawn picture look like what the channel actually
finds? If it does not — which of the two answers will you take to the client, and why?

In [ ]:
# Your code here

## What to take away from this notebook

- **Optimizing the input honestly finds a maximum — and it is useless.** The activation of noise
  turned out to be tens of times above that of any real picture. This is not a breakdown of the
  method but a property of the space: meaningful images occupy a negligible share of it.
- **Regularization is not decoration but a condition of meaning.** Without it the method gives
  an adversarial example. With the shifts the activation drops, and that is the right price: we
  are looking for a maximum inside the region of plausible pictures, not over the whole space.
- **The picture depends on the procedure.** Change the number of steps, the learning rate, the
  weight of the penalty or the starting noise — and you get a different result. We see one of the
  maxima, selected by our own constraints, not "the true image of the channel".
- **One picture per channel is a strong simplification.** If a channel is polysemantic, the
  optimization will give a blend. That is the subject of the lesson "One unit — one concept?".